# 03 Exploratory Analysis

## Goal

Identify, per character, which specific cards are worth building a per-card logistic regression around. Screening pass before modeling: for each (character, card), compare occasions where the card was offered-and-picked against occasions where it was offered-and-passed-over, on:

- **pick rate** — how often the card is taken when offered (popularity, not quality by itself)
- **floors_gained lift** — mean `floors_gained` when picked minus mean `floors_gained` when not picked (does taking this card correlate with the run continuing further from that point?)
- **win_rate lift** — mean `victory` when picked minus mean `victory` when not picked

Source table: `gold_card_choice_events` (one row per card offered per pick-event, `was_picked` flag, `victory`/`floors_gained` as outcome columns). Cards that show up with high lift on both floors_gained and win_rate — and enough sample size to trust the estimate — are the "cards of interest" to carry into per-card logistic regression, per character.

In [ ]:
import os
from pathlib import Path

# Jupyter's cwd is the notebook's directory; project root is one level up.
PROJECT_ROOT = Path.cwd().parent

os.environ["JAVA_HOME"] = str(PROJECT_ROOT / ".jdk17" / "jdk-17.0.20+8")
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = str(PROJECT_ROOT / ".venv" / "Scripts") + os.pathsep + r"C:\hadoop\bin" + os.pathsep + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")
os.environ["PYSPARK_DRIVER_PYTHON"] = str(PROJECT_ROOT / ".venv" / "Scripts" / "python.exe")

GOLD_CARD_CHOICE_EVENTS_PATH = str(PROJECT_ROOT / "raw_data" / "gold" / "card_choice_events")

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (
    SparkSession.builder.master("local[*]")
    .appName("exploratory-analysis")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "100")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(GOLD_CARD_CHOICE_EVENTS_PATH)
print(f"Loaded {GOLD_CARD_CHOICE_EVENTS_PATH}")
print("Row count:", df.count())

## Per-card summary, by character

For every (character, card) pair: how often it's offered, how often it's picked when offered, and the pick/no-pick split on `victory` and `floors_gained`. `character_chosen` already scopes this correctly — a character is only ever offered their own class's cards plus colorless cards, so no extra filtering is needed to keep, e.g., Ironclad-only cards out of the Silent rows.

In [ ]:
card_summary = (
    df.groupBy("character_chosen", "card_name")
    .agg(
        F.count("*").alias("n_offered"),
        F.sum(F.col("was_picked").cast("int")).alias("n_picked"),
        F.mean(F.col("was_picked").cast("double")).alias("pick_rate"),
        F.mean(F.when(F.col("was_picked"), F.col("victory").cast("double"))).alias("win_rate_picked"),
        F.mean(F.when(~F.col("was_picked"), F.col("victory").cast("double"))).alias("win_rate_not_picked"),
        F.mean(F.when(F.col("was_picked"), F.col("floors_gained"))).alias("floors_gained_picked"),
        F.mean(F.when(~F.col("was_picked"), F.col("floors_gained"))).alias("floors_gained_not_picked"),
    )
    .withColumn("win_rate_lift", F.col("win_rate_picked") - F.col("win_rate_not_picked"))
    .withColumn("floors_gained_lift", F.col("floors_gained_picked") - F.col("floors_gained_not_picked"))
)
card_summary.cache()
print("Distinct (character, card) pairs:", card_summary.count())

## Screening: cards where being picked correlates with better outcomes

Restrict to cards with enough sample size in both the picked and not-picked groups to trust the lift estimate (`MIN_GROUP_SIZE` below is a starting guess, not a statistically derived cutoff — tighten it if the top lists still look noisy). Then, per character, show the top cards by `floors_gained_lift` and by `win_rate_lift` side by side — cards appearing high on **both** lists are the strongest "cards of interest" candidates for per-card logistic regression. `pick_rate` is shown alongside for context (popularity isn't evidence of quality by itself, but a high-lift card that's rarely picked is a different story than one that's already being picked constantly).

In [ ]:
MIN_GROUP_SIZE = 500  # minimum n_picked AND n_not_picked to trust a card's lift estimate

screened = card_summary.withColumn("n_not_picked", F.col("n_offered") - F.col("n_picked")) \
    .filter((F.col("n_picked") >= MIN_GROUP_SIZE) & (F.col("n_not_picked") >= MIN_GROUP_SIZE))

print(f"Cards passing the n>={MIN_GROUP_SIZE} screen:", screened.count(), "of", card_summary.count())

screened_pd = screened.select(
    "character_chosen", "card_name", "n_offered", "pick_rate",
    "win_rate_lift", "floors_gained_lift",
).toPandas()

display_cols = ["card_name", "n_offered", "pick_rate", "win_rate_lift", "floors_gained_lift"]
for character in sorted(screened_pd["character_chosen"].unique()):
    char_df = screened_pd[screened_pd["character_chosen"] == character]
    print(f"\n===== {character} — top 15 by floors_gained_lift =====")
    print(char_df.sort_values("floors_gained_lift", ascending=False).head(15)[display_cols].to_string(index=False))
    print(f"\n===== {character} — top 15 by win_rate_lift =====")
    print(char_df.sort_values("win_rate_lift", ascending=False).head(15)[display_cols].to_string(index=False))

## Does popularity track with impact?

Separate question from the top-lists above: across all screened cards, is `pick_rate` actually correlated with `win_rate_lift` / `floors_gained_lift` — i.e. are players already good at picking the cards that help, or are there high-lift cards getting passed over (or popular cards that don't actually move the outcome)? Correlation computed within each character, since pick rates aren't comparable across characters with different card pools.

In [ ]:
corr_rows = []
for character in sorted(screened_pd["character_chosen"].unique()):
    char_df = screened_pd[screened_pd["character_chosen"] == character]
    corr_rows.append({
        "character_chosen": character,
        "n_cards": len(char_df),
        "corr(pick_rate, win_rate_lift)": char_df["pick_rate"].corr(char_df["win_rate_lift"]),
        "corr(pick_rate, floors_gained_lift)": char_df["pick_rate"].corr(char_df["floors_gained_lift"]),
        "corr(win_rate_lift, floors_gained_lift)": char_df["win_rate_lift"].corr(char_df["floors_gained_lift"]),
    })

import pandas as pd
pd.DataFrame(corr_rows)

## Next: per-card logistic regression

Pick the cards of interest from the top lists above (per character) and, for each, fit `victory ~ was_picked + confounders` (HP at pick, floor, relic_count, ascension_level, etc. — all already in `gold_card_choice_events`) restricted to rows where that card was offered. `was_picked`'s coefficient is the card's effect on win probability *controlling for* run state at the time it was offered, which is a cleaner signal than the raw `win_rate_lift` above (that lift doesn't control for anything — a card offered mostly at low HP would look artificially bad). Not built yet — this section is a placeholder for that next pass.

## Stop Spark

Run this when done exploring — otherwise the JVM stays alive holding memory until the kernel is restarted.

In [ ]:
spark.stop()